# NutriScan Model Comparison Runner

Notebook ini menjalankan architecture screen model classifier ringan di Google Colab GPU untuk dataset `nutriscan-mvp-food-dataset-v0.2`.

Sebelum menjalankan notebook:

1. Buka `Runtime > Change runtime type`.
2. Pilih `T4 GPU`.
3. Upload ZIP dataset ke Google Drive, misalnya `MyDrive/nutriscan-mvp-food-dataset-v0.2.zip`.

ZIP dataset harus berisi struktur berikut:

```txt
data/processed-v0.2/
  train/<class>/
  val/<class>/ atau validation/<class>/
  test/<class>/
```

## 1. Verify GPU

Jika `cuda_available` bernilai `False`, ubah runtime Colab ke GPU dulu sebelum lanjut training.

In [ ]:
import torch

print(f"torch={torch.__version__}")
print(f"cuda_available={torch.cuda.is_available()}")
print(f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")

## 2. Configure Paths

Ubah `DATASET_ZIP` jika file ZIP dataset berada di folder Drive lain. Untuk menjalankan kandidat lain, ubah `SELECTED_EXPERIMENT` ke salah satu key di dictionary `EXPERIMENTS`.

In [ ]:
REPO_URL = "https://github.com/BimoAtaullahR/nutri-scan.git"
BRANCH = "ai/baseline"
PROJECT_DIR = "/content/nutri-scan"
AI_DIR = f"{PROJECT_DIR}/services/ai-inference"
DATASET_ZIP = "/content/drive/MyDrive/nutriscan-mvp-food-dataset-v0.2.zip"

SELECTED_EXPERIMENT = "mobilenetv3_large"
EXPERIMENTS = {
    "mobilenetv3_large": {
        "name": "model-comparison-mobilenetv3-large",
        "config": "configs/model_comparison_mobilenetv3_large.json",
    },
    "efficientnet_b2": {
        "name": "model-comparison-efficientnet-b2",
        "config": "configs/model_comparison_efficientnet_b2.json",
    },
    "convnext_tiny": {
        "name": "model-comparison-convnext-tiny",
        "config": "configs/model_comparison_convnext_tiny.json",
    },
    "convnext_tiny_control_label_smoothing": {
        "name": "convnext-tiny-control-label-smoothing",
        "config": "configs/convnext_tiny_control_label_smoothing.json",
    },
    "convnext_tiny_lr5e5": {
        "name": "convnext-tiny-tune-lr5e5",
        "config": "configs/convnext_tiny_tune_lr5e5.json",
    },
    "convnext_tiny_img256": {
        "name": "convnext-tiny-tune-img256",
        "config": "configs/convnext_tiny_tune_img256.json",
    },
    "convnext_tiny_lr5e5_img256": {
        "name": "convnext-tiny-tune-lr5e5-img256",
        "config": "configs/convnext_tiny_tune_lr5e5_img256.json",
    },
}

experiment = EXPERIMENTS[SELECTED_EXPERIMENT]
EXPERIMENT_NAME = experiment["name"]
CONFIG = experiment["config"]
REPORT_DIR = f"reports/{EXPERIMENT_NAME}"
ARTIFACT_DIR = f"model-artifacts/{EXPERIMENT_NAME}"
RESULTS_DIR = f"/content/drive/MyDrive/nutriscan-model-comparison/{EXPERIMENT_NAME}"

print(f"selected={SELECTED_EXPERIMENT}")
print(f"config={CONFIG}")
print(f"report_dir={REPORT_DIR}")
print(f"artifact_dir={ARTIFACT_DIR}")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 4. Clone Repo And Checkout Branch

Cell ini aman dijalankan ulang. Jika folder repo sudah ada, repo akan di-fetch dan checkout branch target.

In [ ]:
import os
import subprocess
from pathlib import Path

def run(command, cwd=None):
    print(f"$ {command}")
    subprocess.run(command, shell=True, cwd=cwd, check=True)

if not Path(PROJECT_DIR).exists():
    run(f"git clone {REPO_URL} {PROJECT_DIR}")

run("git fetch origin --prune", cwd=PROJECT_DIR)
run(f"git checkout {BRANCH}", cwd=PROJECT_DIR)
run(f"git pull --ff-only origin {BRANCH}", cwd=PROJECT_DIR)
run("git status --short --branch", cwd=PROJECT_DIR)

## 5. Extract Dataset

Dataset diextract ke folder `services/ai-inference`, sehingga path akhirnya menjadi `data/processed-v0.2`.

In [ ]:
from pathlib import Path

dataset_zip = Path(DATASET_ZIP)
if not dataset_zip.exists():
    raise FileNotFoundError(f"Dataset ZIP not found: {dataset_zip}")

run(f"unzip -q -o '{DATASET_ZIP}' -d '{AI_DIR}'")
run("find data/processed-v0.2 -maxdepth 2 -type d | sort | head -60", cwd=AI_DIR)

## 6. Sanity Check Dataset Counts

In [ ]:
from pathlib import Path

processed_dir = Path(AI_DIR) / "data/processed-v0.2"
if not processed_dir.exists():
    raise FileNotFoundError(f"Processed dataset folder not found: {processed_dir}")

for split_name in ["train", "val", "validation", "test"]:
    split_dir = processed_dir / split_name
    if not split_dir.exists():
        continue
    print(f"\n{split_name}")
    for class_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
        count = sum(1 for path in class_dir.iterdir() if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
        print(f"  {class_dir.name}: {count}")

## 7. Train Model Comparison Candidate

Cell ini menjalankan script helper:

- install dependencies
- cek CUDA
- dry-run config
- train model dari `CONFIG`
- evaluate predictions
- export misclassified images
- print summary top-1, top-3, dan weak classes

In [ ]:
run(
    f"CONFIG={CONFIG} PROCESSED_DIR=data/processed-v0.2 REQUIRE_CUDA=1 INSTALL_DEPS=1 "
    "bash scripts/colab_retrain_baseline_v2.sh",
    cwd=AI_DIR,
)

## 8. Inspect Metrics

In [ ]:
import json
from pathlib import Path
from IPython.display import Markdown, display

report_dir = Path(AI_DIR) / REPORT_DIR
metrics = json.loads((report_dir / "metrics.json").read_text())
per_class = json.loads((report_dir / "per_class_metrics.json").read_text())

weak_labels = ["rendang", "gado_gado", "soto"]
weak_classes = {label: per_class[label] for label in weak_labels}
weak_class_avg_f1 = sum(values["f1"] for values in weak_classes.values()) / len(weak_classes)

summary = {
    "experiment_name": EXPERIMENT_NAME,
    "config": CONFIG,
    "report_dir": REPORT_DIR,
    "artifact_dir": ARTIFACT_DIR,
    "top1_accuracy": metrics["top1_accuracy"],
    "top3_accuracy": metrics["top3_accuracy"],
    "num_test_samples": metrics["num_test_samples"],
    "meets_mvp_target": metrics["meets_mvp_target"],
    "weak_class_avg_f1": weak_class_avg_f1,
    "weak_classes": weak_classes,
}

def pct(value):
    return f"{value * 100:.2f}%"

summary_md = f"""
## Model Comparison Run

| Field | Value |
|---|---|
| Experiment | `{EXPERIMENT_NAME}` |
| Config | `{CONFIG}` |
| Report Dir | `{REPORT_DIR}` |
| Artifact Dir | `{ARTIFACT_DIR}` |
| Top-1 | {pct(metrics['top1_accuracy'])} |
| Top-3 | {pct(metrics['top3_accuracy'])} |
| Test Samples | {metrics['num_test_samples']} |
| MVP Passed | {metrics['meets_mvp_target']} |
| Weak-class Avg F1 | {pct(weak_class_avg_f1)} |

## Weak Classes

| Class | Precision | Recall | F1 | Support |
|---|---:|---:|---:|---:|
"""

for label, values in weak_classes.items():
    summary_md += (
        f"| {label} | {pct(values['precision'])} | {pct(values['recall'])} | "
        f"{pct(values['f1'])} | {values['support']} |\n"
    )

display(Markdown(summary_md))
(report_dir / "model_comparison_summary.md").write_text(summary_md.strip() + "\n")
(report_dir / "model_comparison_summary.json").write_text(json.dumps(summary, indent=2) + "\n")

## 9. Save Results Back To Google Drive

Output ini yang perlu kamu ambil untuk review dan sharing. Jangan commit model artifact, dataset, atau generated reports ke Git.

In [ ]:
run(f"mkdir -p '{RESULTS_DIR}'")
run(f"cp -r {ARTIFACT_DIR} '{RESULTS_DIR}/'", cwd=AI_DIR)
run(f"cp -r {REPORT_DIR} '{RESULTS_DIR}/'", cwd=AI_DIR)
run(f"find '{RESULTS_DIR}' -maxdepth 3 -type f | sort | head -80")

## 10. Automated ConvNeXt Tuning Batch

Jalankan cell ini kalau ingin menjalankan seluruh batch tuning ConvNeXt secara berurutan. Jangan jalankan cell single-run di atas pada saat yang sama.

In [ ]:
import shlex

TUNING_SEQUENCE = [
    "convnext_tiny_control_label_smoothing",
    "convnext_tiny_lr5e5",
    "convnext_tiny_img256",
    "convnext_tiny_lr5e5_img256",
]

for index, selected in enumerate(TUNING_SEQUENCE, start=1):
    experiment = EXPERIMENTS[selected]
    experiment_name = experiment["name"]
    config = experiment["config"]
    report_dir = f"reports/{experiment_name}"
    artifact_dir = f"model-artifacts/{experiment_name}"
    results_dir = f"/content/drive/MyDrive/nutriscan-model-comparison/{experiment_name}"
    install_deps = "1" if index == 1 else "0"

    print(f"\n=== [{index}/{len(TUNING_SEQUENCE)}] {selected} ===")
    print(f"config={config}")
    print(f"report_dir={report_dir}")
    print(f"artifact_dir={artifact_dir}")

    run(
        f"CONFIG={shlex.quote(config)} "
        f"PROCESSED_DIR=data/processed-v0.2 "
        f"REQUIRE_CUDA=1 "
        f"INSTALL_DEPS={install_deps} "
        "bash scripts/colab_retrain_baseline_v2.sh",
        cwd=AI_DIR,
    )

    run(f"mkdir -p {shlex.quote(results_dir)}")
    run(f"cp -r {shlex.quote(artifact_dir)} {shlex.quote(results_dir)}/", cwd=AI_DIR)
    run(f"cp -r {shlex.quote(report_dir)} {shlex.quote(results_dir)}/", cwd=AI_DIR)

print("\nAll ConvNeXt tuning runs completed.")

## Output Penting

Hasil utama setelah training:

```txt
{ARTIFACT_DIR}/model.pt
{ARTIFACT_DIR}/label_map.json
{REPORT_DIR}/predictions.json
{REPORT_DIR}/metrics.json
{REPORT_DIR}/per_class_metrics.json
{REPORT_DIR}/confusion_matrix.json
{REPORT_DIR}/model_comparison_summary.md
{REPORT_DIR}/model_comparison_summary.json
{REPORT_DIR}/misclassified/
```

Cell `Inspect Metrics` otomatis menampilkan ringkasan eksperimen aktif:

- top-1 accuracy
- top-3 accuracy
- weak-class average F1
- weak classes: `rendang`, `gado_gado`, `soto`